In [1]:
from pyspark.sql import *

In [2]:
from pyspark.sql.functions import *

In [3]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

spark = SparkSession. \
    builder. \
    master('yarn'). \
    config('spark.ui.port', '0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    getOrCreate()

In [4]:
data = [
    ("U014", "liam@email.com", 29),
    ("U015", "emma@email.com", 31),
    ("U020", "olivia@email.com", 27),
    ("U023", "william@email.com", 50),
    ("U026", "isabella@email.com", 33),
    ("U027", "sophia@email.com", 24),
    (None, "no_id@email.com", 25),
    (None, "null_again@email.com", 55),
    ("U016", "bad_email.com", 40),
    ("U021", "no_domain@", 35),
    ("U022", "spaces in@email.com", 45),
    ("U017", "negative@email.com", -15),
    ("U018", "old@email.com", 130),
    ("U024", "james@email.com", 121),
    ("U019", "dupe_group@email.com", 22),
    ("U019", "dupe_group@email.com", 22),
    ("U019", "dupe_group@email.com", 22),
    ("U025", "lucas@email.com", 28),
    ("U025", "lucas@email.com", 28),
    (None, "broken everything", -99)
]

In [5]:
columns = ["cust_id", "email", "age"]

In [6]:
raw_silver_df = spark.createDataFrame(data, columns)

In [7]:
raw_silver_df.show()

+-------+--------------------+---+
|cust_id|               email|age|
+-------+--------------------+---+
|   U014|      liam@email.com| 29|
|   U015|      emma@email.com| 31|
|   U020|    olivia@email.com| 27|
|   U023|   william@email.com| 50|
|   U026|  isabella@email.com| 33|
|   U027|    sophia@email.com| 24|
|   null|     no_id@email.com| 25|
|   null|null_again@email.com| 55|
|   U016|       bad_email.com| 40|
|   U021|          no_domain@| 35|
|   U022| spaces in@email.com| 45|
|   U017|  negative@email.com|-15|
|   U018|       old@email.com|130|
|   U024|     james@email.com|121|
|   U019|dupe_group@email.com| 22|
|   U019|dupe_group@email.com| 22|
|   U019|dupe_group@email.com| 22|
|   U025|     lucas@email.com| 28|
|   U025|     lucas@email.com| 28|
|   null|   broken everything|-99|
+-------+--------------------+---+



# DQ Check 1


In [8]:
raw_silver_df = raw_silver_df.withColumnRenamed("cust_id","customer_id")

In [9]:
raw_silver_df.show()

+-----------+--------------------+---+
|customer_id|               email|age|
+-----------+--------------------+---+
|       U014|      liam@email.com| 29|
|       U015|      emma@email.com| 31|
|       U020|    olivia@email.com| 27|
|       U023|   william@email.com| 50|
|       U026|  isabella@email.com| 33|
|       U027|    sophia@email.com| 24|
|       null|     no_id@email.com| 25|
|       null|null_again@email.com| 55|
|       U016|       bad_email.com| 40|
|       U021|          no_domain@| 35|
|       U022| spaces in@email.com| 45|
|       U017|  negative@email.com|-15|
|       U018|       old@email.com|130|
|       U024|     james@email.com|121|
|       U019|dupe_group@email.com| 22|
|       U019|dupe_group@email.com| 22|
|       U019|dupe_group@email.com| 22|
|       U025|     lucas@email.com| 28|
|       U025|     lucas@email.com| 28|
|       null|   broken everything|-99|
+-----------+--------------------+---+



In [10]:
My_Window = Window.partitionBy("customer_id")

In [11]:
df_with_dupes = raw_silver_df.withColumn("dupes_count",count("customer_id").over(My_Window))

In [12]:
df_with_dupes.show(10)

+-----------+--------------------+---+-----------+
|customer_id|               email|age|dupes_count|
+-----------+--------------------+---+-----------+
|       U014|      liam@email.com| 29|          1|
|       U024|     james@email.com|121|          1|
|       null|     no_id@email.com| 25|          0|
|       null|null_again@email.com| 55|          0|
|       null|   broken everything|-99|          0|
|       U027|    sophia@email.com| 24|          1|
|       U016|       bad_email.com| 40|          1|
|       U025|     lucas@email.com| 28|          2|
|       U025|     lucas@email.com| 28|          2|
|       U017|  negative@email.com|-15|          1|
+-----------+--------------------+---+-----------+
only showing top 10 rows



# DQ Checks 2,3,4,5


In [13]:

dq_evaluated_df = df_with_dupes.withColumn(
    "dq_status",
    
    when(col("customer_id").isNull(), "FAIL: Null customer_id")
    
    .when(~col("email").rlike(r"^[\w\.-]+@[\w\.-]+\.\w+$"), "FAIL: Incorrect Email Format")
    
    .when((col("age").cast("int") < 0) | (col("age").cast("int") > 120), "FAIL: Age Out of Range")
    
    .when(col("dupes_count") > 1, "FAIL: Duplicate Record")
    
    .otherwise("PASS")
)


In [14]:
dq_evaluated_df.show()

+-----------+--------------------+---+-----------+--------------------+
|customer_id|               email|age|dupes_count|           dq_status|
+-----------+--------------------+---+-----------+--------------------+
|       U014|      liam@email.com| 29|          1|                PASS|
|       U024|     james@email.com|121|          1|FAIL: Age Out of ...|
|       null|     no_id@email.com| 25|          0|FAIL: Null custom...|
|       null|null_again@email.com| 55|          0|FAIL: Null custom...|
|       null|   broken everything|-99|          0|FAIL: Null custom...|
|       U027|    sophia@email.com| 24|          1|                PASS|
|       U016|       bad_email.com| 40|          1|FAIL: Incorrect E...|
|       U025|     lucas@email.com| 28|          2|FAIL: Duplicate R...|
|       U025|     lucas@email.com| 28|          2|FAIL: Duplicate R...|
|       U017|  negative@email.com|-15|          1|FAIL: Age Out of ...|
|       U019|dupe_group@email.com| 22|          3|FAIL: Duplicat

In [15]:
dq_evaluated_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- age: long (nullable = true)
 |-- dupes_count: long (nullable = false)
 |-- dq_status: string (nullable = false)



In [19]:
gold_df = dq_evaluated_df.filter(col("dq_status") == "PASS").drop("dupes_count", "dq_status")

In [20]:
gold_df.show()


+-----------+------------------+---+
|customer_id|             email|age|
+-----------+------------------+---+
|       U014|    liam@email.com| 29|
|       U027|  sophia@email.com| 24|
|       U015|    emma@email.com| 31|
|       U026|isabella@email.com| 33|
|       U023| william@email.com| 50|
|       U020|  olivia@email.com| 27|
+-----------+------------------+---+



In [21]:
quarantine_df = dq_evaluated_df.filter(col("dq_status") != "PASS") \
                               .withColumn("quarantine_date", current_date())

In [22]:
quarantine_df.show()

+-----------+--------------------+---+-----------+--------------------+---------------+
|customer_id|               email|age|dupes_count|           dq_status|quarantine_date|
+-----------+--------------------+---+-----------+--------------------+---------------+
|       U024|     james@email.com|121|          1|FAIL: Age Out of ...|     2026-05-04|
|       null|     no_id@email.com| 25|          0|FAIL: Null custom...|     2026-05-04|
|       null|null_again@email.com| 55|          0|FAIL: Null custom...|     2026-05-04|
|       null|   broken everything|-99|          0|FAIL: Null custom...|     2026-05-04|
|       U016|       bad_email.com| 40|          1|FAIL: Incorrect E...|     2026-05-04|
|       U025|     lucas@email.com| 28|          2|FAIL: Duplicate R...|     2026-05-04|
|       U025|     lucas@email.com| 28|          2|FAIL: Duplicate R...|     2026-05-04|
|       U017|  negative@email.com|-15|          1|FAIL: Age Out of ...|     2026-05-04|
|       U019|dupe_group@email.co

## DQ Reports

In [30]:
quarantine_df.select("customer_id", "email", "age", "dq_status", "quarantine_date").show(truncate=False)

+-----------+--------------------+---+----------------------------+---------------+
|customer_id|email               |age|dq_status                   |quarantine_date|
+-----------+--------------------+---+----------------------------+---------------+
|U024       |james@email.com     |121|FAIL: Age Out of Range      |2026-05-04     |
|null       |no_id@email.com     |25 |FAIL: Null customer_id      |2026-05-04     |
|null       |null_again@email.com|55 |FAIL: Null customer_id      |2026-05-04     |
|null       |broken everything   |-99|FAIL: Null customer_id      |2026-05-04     |
|U016       |bad_email.com       |40 |FAIL: Incorrect Email Format|2026-05-04     |
|U025       |lucas@email.com     |28 |FAIL: Duplicate Record      |2026-05-04     |
|U025       |lucas@email.com     |28 |FAIL: Duplicate Record      |2026-05-04     |
|U017       |negative@email.com  |-15|FAIL: Age Out of Range      |2026-05-04     |
|U019       |dupe_group@email.com|22 |FAIL: Duplicate Record      |2026-05-0

In [25]:
dq_report_df = quarantine_df.groupBy("dq_status", "quarantine_date") \
                            .agg(count("*").alias("failure_count")) \
                            .orderBy(desc("failure_count"))

In [31]:
dq_report_df.show(truncate=False)

+----------------------------+---------------+-------------+
|dq_status                   |quarantine_date|failure_count|
+----------------------------+---------------+-------------+
|FAIL: Duplicate Record      |2026-05-04     |5            |
|FAIL: Null customer_id      |2026-05-04     |3            |
|FAIL: Age Out of Range      |2026-05-04     |3            |
|FAIL: Incorrect Email Format|2026-05-04     |3            |
+----------------------------+---------------+-------------+



In [34]:
quarantine_df.filter(col("dq_status") == "FAIL: Incorrect Email Format") \
             .select("customer_id", "email", "dq_status") \
             .limit(1) \
             .show(truncate=False)

+-----------+-------------+----------------------------+
|customer_id|email        |dq_status                   |
+-----------+-------------+----------------------------+
|U016       |bad_email.com|FAIL: Incorrect Email Format|
+-----------+-------------+----------------------------+

